
# Прогноз виручки за 7 днів для оцінки маркетингу.
**Розроблений прогноз має допомагати маркетологам якомога раніше і точніше приймати рішення**

**Дані:**
- `task_2_users_params.csv` — параметри користувачів при реєстрації + таргет `revenue_7d`
- `task_2_users_actions.csv` — дії користувачів за перший день (4 інтервали по 6 годин)

In [1]:
# Libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import ylabel

In [2]:
# Зчитуємо дані та оглядаємо
actions = pd.read_csv("../../data/task_2_users_actions.csv")
params = pd.read_csv("../../data/task_2_users_params.csv")

------------------
# EDA



In [3]:
df_actions = actions.copy()
df_params = params.copy()

In [4]:
print("task_2_users_actions.csv: ")
print("Shape: ", df_actions.shape)
display(df_actions.head(3))

print("task_2_users_params.csv: ")
print("Shape: ", df_params.shape)
display(df_params.head(3))

task_2_users_actions.csv: 
Shape:  (6456848, 8)


,id_user,timestamp_interval_start,timestamp_interval_end,sum_payments,cnt_payments,sum_credits_spend,cnt_returns,cnt_visit_other_users
0,2672114,2024-08-20 07:07:14.493905+00:00,2024-08-20 13:07:14.493905+00:00,0.0,0,0,0,2
1,2672114,2024-08-20 13:07:14.493905+00:00,2024-08-20 19:07:14.493905+00:00,0.0,0,0,0,0
2,2672114,2024-08-20 19:07:14.493905+00:00,2024-08-21 01:07:14.493905+00:00,0.0,0,0,0,0


task_2_users_params.csv: 
Shape:  (1905428, 13)


,id_user,timestamp_reg,age,traffic_type_id,traffic_group_id,gender,country,device_browser,device_brand,device_model,device_platform,device_os,revenue_7d
0,1,2024-09-28 01:32:16.011076+00:00,53,1,1,male,CA,Mobile Silk,Amazon,Fire HD 10 (2021),tablet,Android,0.0
1,1,2024-09-28 01:32:16.011076+00:00,53,1,1,male,CA,Mobile Silk,Amazon,Fire HD 10 (2021),tablet,Android,0.0
2,13,2024-10-18 17:05:48.437175+00:00,53,1,7,male,US,Mobile Silk,Amazon,Fire HD 8 (2022),tablet,Android,0.0


In [5]:
# Детальна інформація по датасетам
df_actions.info(memory_usage='deep')
df_params.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 6456848 entries, 0 to 6456847
Data columns (total 8 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   id_user                   int64  
 1   timestamp_interval_start  str    
 2   timestamp_interval_end    str    
 3   sum_payments              float64
 4   cnt_payments              int64  
 5   sum_credits_spend         int64  
 6   cnt_returns               int64  
 7   cnt_visit_other_users     int64  
dtypes: float64(1), int64(5), str(2)
memory usage: 1.3 GB
<class 'pandas.DataFrame'>
RangeIndex: 1905428 entries, 0 to 1905427
Data columns (total 13 columns):
 #   Column            Dtype  
---  ------            -----  
 0   id_user           int64  
 1   timestamp_reg     str    
 2   age               int64  
 3   traffic_type_id   int64  
 4   traffic_group_id  int64  
 5   gender            str    
 6   country           str    
 7   device_browser    str    
 8   device_brand      str    
 9   dev

In [6]:
miss_data_in_params = df_params.isnull().sum()
miss_data_in_actions = df_actions.isnull().sum()

print("Miss data(users_actions):\n",miss_data_in_params,'\n')
print("Miss data(users_params):\n",miss_data_in_actions, '\n')

print("Params shape:", df_params.shape)
print("Actions shape:", df_actions.shape)

print("Unique users in pairs:", df_params['id_user'].nunique())
print("Unique users in actions:", df_actions['id_user'].nunique())

Miss data(users_actions):
 id_user                  0
timestamp_reg            0
age                      0
traffic_type_id          0
traffic_group_id         0
gender                   0
country               4345
device_browser        6861
device_brand        283445
device_model        181811
device_platform       6893
device_os             6861
revenue_7d               0
dtype: int64 

Miss data(users_params):
 id_user                     0
timestamp_interval_start    0
timestamp_interval_end      0
sum_payments                0
cnt_payments                0
sum_credits_spend           0
cnt_returns                 0
cnt_visit_other_users       0
dtype: int64 

Params shape: (1905428, 13)
Actions shape: (6456848, 8)
Унікальних юзерів в params: 1614200
Унікальних юзерів в actions: 1614200


In [7]:
df_params[['age', 'revenue_7d']].describe()
df_actions[['sum_payments', 'cnt_payments', 'sum_credits_spend']].describe()

,sum_payments,cnt_payments,sum_credits_spend
count,6.456848e+06,6.456848e+06,6.456848e+06
mean,1.032068e-01,7.044459e-03,3.716503e+00
std,3.940898e+00,1.431226e-01,2.475382e+01
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00
max,2.446800e+03,5.500000e+01,1.573000e+04


In [8]:
print(df_params['gender'].value_counts())
print(df_params['device_platform'].value_counts())

gender
male      1667485
female     237943
Name: count, dtype: int64
device_platform
mobile     1713627
desktop     145571
tablet       38154
tv             850
console        332
car              1
Name: count, dtype: int64


In [9]:
# Таргет — сильно скошений, більшість = 0
print('Всього рядків:', len(params))
print('revenue_7d == 0:', (params['revenue_7d'] == 0).sum())
print('revenue_7d > 0:', (params['revenue_7d'] > 0).sum())
print()
print(params['revenue_7d'].describe())

Всього рядків: 1905428
revenue_7d == 0: 1865898
revenue_7d > 0: 39530

count    1.905428e+06
mean     1.099873e+00
std      3.259617e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.040958e+04
Name: revenue_7d, dtype: float64


Велика кількість нулів, наша регресійна модель не зможе прогнозувати, бо багато однакових значень, так наша модель буде більше тягнути прогнозовані дані до нуля.

In [10]:
# Топ країни за середньою виручкою
country_revenue = (
    params.groupby('country')['revenue_7d']
    .agg(['mean', 'sum', 'count'])
    .sort_values('mean', ascending=False)
    .head(15)
)
country_revenue

,mean,sum,count
country,,,
NF,64.980000,64.98,1
GP,10.106928,1546.36,153
MC,9.360800,234.02,25
GU,5.388336,2979.75,553
PF,4.319393,924.35,214
IS,3.419638,2643.38,773
CH,3.051354,6041.68,1980
TC,2.783793,322.92,116
NO,2.520922,8964.40,3556


In [11]:
# Revenue по traffic_type
params.groupby('traffic_type_id')['revenue_7d'].agg(['mean', 'sum', 'count'])

,mean,sum,count
traffic_type_id,,,
1,0.803578,795105.40,989457
2,1.470265,1187296.26,807539
3,1.045147,113327.38,108432


In [12]:
# Revenue по device_platform
params.groupby('device_platform')['revenue_7d'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)

,mean,sum,count
device_platform,,,
desktop,1.849720,269265.61,145571
tablet,1.367464,52174.24,38154
mobile,1.029381,1763975.48,1713627
tv,0.035412,30.10,850
console,0.019578,6.50,332
car,0.000000,0.00,1


-------------------
# Feature Engineering

In [13]:
# Переведемо часові фічі в потрібний нам формат
df_params['timestamp_reg'] = pd.to_datetime(
    df_params['timestamp_reg'],
    format='mixed'
)

df_actions['timestamp_interval_start'] = pd.to_datetime(
    df_actions['timestamp_interval_start'],
    format='mixed'
)

df_actions['timestamp_interval_end'] = pd.to_datetime(
    df_actions['timestamp_interval_end'],
    format='mixed'
)

In [14]:
# Видалення дублікатів.
print("Duplicate: ", df_params.duplicated().sum())
print("Duplicate: ", df_actions.duplicated().sum())

df_actions = df_actions.drop_duplicates()
df_params = df_params.sort_values('timestamp_reg').drop_duplicates(subset='id_user', keep='first')

print("After delete duplicate: ", df_params.duplicated().sum())
print("After delete duplicate: ", df_actions.duplicated().sum())
print("Now lines in params:", df_params.shape[0])

Duplicate:  291127
Duplicate:  39
After delete duplicate:  0
After delete duplicate:  0
Now lines in params: 1614200


In [15]:
df_actions.head(8)

,id_user,timestamp_interval_start,timestamp_interval_end,sum_payments,cnt_payments,sum_credits_spend,cnt_returns,cnt_visit_other_users
0,2672114,2024-08-20 07:07:14.493905+00:00,2024-08-20 13:07:14.493905+00:00,0.0,0,0,0,2
1,2672114,2024-08-20 13:07:14.493905+00:00,2024-08-20 19:07:14.493905+00:00,0.0,0,0,0,0
2,2672114,2024-08-20 19:07:14.493905+00:00,2024-08-21 01:07:14.493905+00:00,0.0,0,0,0,0
3,2672114,2024-08-21 01:07:14.493905+00:00,2024-08-21 07:07:14.493905+00:00,0.0,0,0,0,0
4,4314012,2024-08-20 07:07:25.610218+00:00,2024-08-20 13:07:25.610218+00:00,0.0,0,0,0,4
5,4314012,2024-08-20 13:07:25.610218+00:00,2024-08-20 19:07:25.610218+00:00,0.0,0,0,0,0
6,4314012,2024-08-20 19:07:25.610218+00:00,2024-08-21 01:07:25.610218+00:00,0.0,0,0,0,0
7,4314012,2024-08-21 01:07:25.610218+00:00,2024-08-21 07:07:25.610218+00:00,0.0,0,0,0,0


In [16]:
# Агрегуємо actions по юзеру
# Кожен юзер має 4 інтервали (6-годинні) => агрегуємо суми та розраховуємо поведінкові фічі
actions_agg = df_actions.groupby('id_user').agg(
    act_sum_payments=('sum_payments', 'sum'),
    act_max_payments=('sum_payments', 'max'),
    act_cnt_payments=('cnt_payments', 'sum'),
    act_sum_credits=('sum_credits_spend', 'sum'),
    act_cnt_returns=('cnt_returns', 'sum'),
    act_sum_visits=('cnt_visit_other_users', 'sum'),
    act_max_visits=('cnt_visit_other_users', 'max'),
    # Чи була активність взагалі у другому/третьому/четвертому інтервалі
    act_active_intervals=('cnt_visit_other_users', lambda x: (x > 0).sum()),
).reset_index()

print(actions_agg.shape)
actions_agg.head()

(1614200, 9)


,id_user,act_sum_payments,act_max_payments,act_cnt_payments,act_sum_credits,act_cnt_returns,act_sum_visits,act_max_visits,act_active_intervals
0,1,0.0,0.0,0,0,0,0,0,0
1,13,0.0,0.0,0,0,0,1,1,1
2,15,0.0,0.0,0,0,0,0,0,0
3,16,0.0,0.0,0,160,0,0,0,0
4,19,0.0,0.0,0,101,0,0,0,0


In [30]:
# З'єднуємо дві таблиці по ключу "id_user". (LEFT JOIN)
df = df_params.merge(actions_agg, on='id_user', how='left')

-----------------
# Підготовка для моделювання. Обробка фінального датасету

In [31]:
print('Shape:', df.shape, '\n')
print('Miss data:\n', df.isnull().sum(), '\n')
print('Duplicate: ', df.duplicated().sum())
display(df.head(3))

Shape: (1614200, 21) 

Miss data:
 id_user                      0
timestamp_reg                0
age                          0
traffic_type_id              0
traffic_group_id             0
gender                       0
country                   3790
device_browser            6861
device_brand            248046
device_model            167566
device_platform           6887
device_os                 6861
revenue_7d                   0
act_sum_payments             0
act_max_payments             0
act_cnt_payments             0
act_sum_credits              0
act_cnt_returns              0
act_sum_visits               0
act_max_visits               0
act_active_intervals         0
dtype: int64 

Duplicate:  0


,id_user,timestamp_reg,age,traffic_type_id,traffic_group_id,gender,country,device_browser,device_brand,device_model,...,device_os,revenue_7d,act_sum_payments,act_max_payments,act_cnt_payments,act_sum_credits,act_cnt_returns,act_sum_visits,act_max_visits,act_active_intervals
0,33861,2024-08-01 00:00:01.423192+00:00,23,1,1,male,NZ,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0,0,0,1,1,1
1,67459,2024-08-01 00:00:05.687805+00:00,21,1,7,male,US,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0,127,0,8,6,2
2,174142,2024-08-01 00:00:06.062047+00:00,23,1,1,male,US,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0,0,0,1,1,1
